# Full-Packet Partial Stress Viewer

**Partial/debug stress-test only.** This notebook reads generated quick-analysis outputs. It does not loop over raw files, expand cutouts, or make completeness claims.

Not a complete week. Not a complete first day. Not a validated all-alert census.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, Image, display

OUT_DIR = Path('../outputs/data_transfer/quick_analysis/full_week_full_packet_partial').resolve()
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR

## Dataset Status And Limitation Banner

In [ ]:
metadata_path = OUT_DIR / 'raw_metadata_summary.json'
sample_summary_candidates = sorted(OUT_DIR.glob('sample_*_summary.json'))
metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
sample_summary = json.loads(sample_summary_candidates[-1].read_text()) if sample_summary_candidates else {}
display(Markdown('**Completeness claim:** blocked. This is partial/debug stress testing only.'))
pd.DataFrame([{
    'raw_files': metadata.get('file_count'),
    'readable_files': metadata.get('readable_file_count'),
    'readable_rows': metadata.get('readable_rows'),
    'expected_total': metadata.get('expected_total'),
    'apparent_percent_complete': metadata.get('apparent_percent_complete'),
    'schema_groups': metadata.get('schema_group_count'),
    'scientific_completeness': metadata.get('scientific_completeness'),
}])

## Field Availability

In [ ]:
columns = sample_summary.get('sample_columns') or []
pd.DataFrame({'column': columns})

## Flattened Outputs And Batch Files

In [ ]:
combined_candidates = sorted(OUT_DIR.glob('sample_*_files_flattened.parquet'))
batch_files = sorted((OUT_DIR / 'batches').glob('flattened_batch_*.parquet')) if (OUT_DIR / 'batches').exists() else []
sample_path = combined_candidates[-1] if combined_candidates else None
sample = pd.read_parquet(sample_path) if sample_path and sample_path.exists() else pd.DataFrame()
{'combined_file': str(sample_path) if sample_path else None, 'batch_count': len(batch_files), 'sample_rows_loaded': len(sample)}

## Band Distribution

In [ ]:
pd.DataFrame({
    'band': list((sample_summary.get('band_counts') or {}).keys()),
    'rows': list((sample_summary.get('band_counts') or {}).values()),
})

## Object/Source Repetition

In [ ]:
{
    'object_column': sample_summary.get('object_column'),
    'source_column': sample_summary.get('source_column'),
    'unique_objects': sample_summary.get('unique_objects'),
    'unique_sources': sample_summary.get('unique_sources'),
    'diaObjectId_zero_warning': 'diaObjectId=0 is not interpreted as a real repeated object population',
}

## Classifier, Time, Sky, And Missingness

In [ ]:
{
    'classifier_columns': sample_summary.get('classifier_columns'),
    'time_coverage': sample_summary.get('time_coverage'),
    'sky_coverage': sample_summary.get('sky_coverage'),
    'sesar_style_summary': sample_summary.get('sesar_style_summary'),
}

In [ ]:
missingness = sample_summary.get('missingness') or {}
pd.Series(missingness).sort_values(ascending=False).head(40)

## Figure Gallery

In [ ]:
figure_files = sorted(FIG_DIR.glob('*.png')) if FIG_DIR.exists() else []
for path in figure_files:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))
len(figure_files)

## What This Supports / Does Not Support

Supports: schema stress testing, field availability review, partial data-quality checks, plotting ergonomics, classifier sanity checks, and full-packet performance planning.

Does not support: full-week completeness, first-day completeness, all-alert census completeness, variable-source census completeness, or astrophysical population claims.

## Next Scientific Steps

- Finish/resume the full Data Transfer download before completeness validation.
- Add selective nested serialization for full-packet ingestion.
- Preserve cutout/heavy fields in sidecars if needed, not inline quick-analysis tables.
- Use complete-night manifests for population-level analysis only after strict validation.